$$\Huge\mbox{PUC Minas}$$

$$\large\mbox{Séries Temporais}$$

$$\small\mbox{Prof. Leopoldo Grajeda}$$

# Séries temporais sazonais: AutoARIMA

Este notebook busca uma série temporal de um arquivo escolhido a aplica os modelos SARIMA com seleção automática de $(p,d,q)(P,D,Q)_m$.

Data da última atualização: 17/03/2025

#### Preliminares

In [ ]:
# Carregamento das bibliotecas

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt


In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.arima import arima_string

In [ ]:
# Ajuste das preferências gráficas

plt.style.use('fivethirtyeight')
plt.rcParams['lines.linewidth'] = 1.5
dark_style = {
    'figure.facecolor': '#FFFFFF',    # Cor da moldura
    'axes.facecolor': '#FFFFFF',      # Cor do fundo
    'savefig.facecolor':'#000000',
    'axes.grid': True,
    'axes.grid.which': 'both',
    'axes.spines.left': False,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.spines.bottom': True,
    'grid.color': '#888888',
    'grid.linewidth': '0.1',
    'text.color': '#000000',
    'axes.labelcolor': '#000000',
    'xtick.color': '#000000',
    'ytick.color': '#000000',
    'font.size': 12 }
plt.rcParams.update(dark_style)

from pylab import rcParams
rcParams['figure.figsize'] = (21,13)

#### Carregamento da Série Temporal

In [ ]:
# Leitura do arquivo de dados
MinhaSerieTemporal = pd.read_csv('Passageiros.csv', index_col = 0)

# Ajuste do índice para formato DateTime
MinhaSerieTemporal.index = pd.to_datetime(MinhaSerieTemporal.index)

# Exibição do DataFrame
MinhaSerieTemporal

In [ ]:
# Gráfico
MinhaSerieTemporal.plot()

#### Separação das bases de treino e teste

In [ ]:
# Definição da proporção de dados para compor a base de teste

PercentualTeste = 10

# Cálculo do tamanho do período de testes
PeriodoTeste = PercentualTeste * len(MinhaSerieTemporal) // 100

# Definição do DataFrame de treino
TreinoDF = pd.DataFrame(index = MinhaSerieTemporal[:-PeriodoTeste].index)
TreinoDF['Treino'] = MinhaSerieTemporal[:-PeriodoTeste][MinhaSerieTemporal.columns[0]]

# Definição do DataFrame de teste
TesteDF  = pd.DataFrame(index = MinhaSerieTemporal[-PeriodoTeste:].index)
TesteDF['Teste'] = MinhaSerieTemporal[-PeriodoTeste:][MinhaSerieTemporal.columns[0]]

# Exibe a série temporal, com a separação da base de dados
pd.concat([TreinoDF,TesteDF], axis = 1).plot()

#### Previsão pelo algoritmo AutoARIMA

In [ ]:

# Formatação do DataFrame no padrão do StatsForecast
SerieSF = pd.DataFrame()
SerieSF['ds'] = TreinoDF .index.values
SerieSF['y']  = TreinoDF [TreinoDF.columns[0]].values
SerieSF['unique_id'] = TreinoDF.columns[0]

SerieSF

In [ ]:
season_length = 12        # anual

InstanteInicial = dt.datetime.now()

# Realiza o ajuste do modelo escolhido
models = [AutoARIMA(season_length=season_length)]
sf = StatsForecast(models=models, freq='ME')
sf.fit(df=SerieSF)

InstanteFinal = dt.datetime.now()

print(f'Modelo: {arima_string(sf.fitted_[0,0].model_)}')
print(f'Tempo:  {str(InstanteFinal - InstanteInicial)}')

In [ ]:

# Define o horizonte futuro de previsões out-of-sample
HorizontePrev = 60

# Realiza as previsões com o modelo ajustado
sf.predict(h = HorizontePrev)
